# Required Assignment 31
## Building and Evaluating NLP Applications Using Hugging Face Transformers
**Cheah Sze Zheng**

Completed from the supplied Emeritus notebook. The original sentiment sentences and
sequence-classification sentence are retained; four additional pipeline examples are added.
This is an executed tutorial, not a benchmark of deployment accuracy. Exact outputs,
package versions and model revision identifiers are saved below.

## Observed results

The two supplied sentiment examples were labelled POSITIVE (0.9997) and NEGATIVE (0.9994). The mixed-sentiment probe was NEGATIVE (0.9980); the sarcastic crash complaint was incorrectly labelled POSITIVE (0.9729), illustrating that a high score can accompany an error.

Fill-mask ranked “capital” first (0.9815). Named entity recognition identified Sarah as PER, Microsoft as ORG and London as LOC. Extractive question answering returned “Singapore” (0.9910), and its character span matched the supplied context. These are simple demonstrations, not estimates of general accuracy.

Greedy DistilGPT2 generation repeatedly produced “learn how to use the data”, giving an unhelpful continuation. This is an observed repetition failure; the example does not establish a hallucination rate.

The base model returned hidden states of shape [2, 7, 768]. The sequence classifier labelled “This course is excellent.” POSITIVE (0.999862) and “Awful.” NEGATIVE (0.999797). Both Softmax rows summed to one within numerical tolerance. The shorter sentence had three padding tokens, each masked with zero.

## Setup
Python 3.12. To reproduce in a fresh environment:
```bash
python -m pip install torch==2.9.1 transformers==4.57.1 huggingface-hub==0.36.2 safetensors==0.8.0 nbformat nbclient ipykernel
```
CPU execution downloads public pretrained weights on first use. No API key or customer data is required.
All text inputs remain local during inference. Cache files are stored in the working directory.

In [1]:
import os, json, platform, importlib.metadata, gc, warnings
from pathlib import Path
os.environ['HF_HOME'] = str(Path.cwd() / '_model_cache')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# Suppress optional notebook progress-widget warnings, not model warnings.
warnings.filterwarnings('ignore', message='IProgress not found.*')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification, set_seed
from huggingface_hub import HfApi
torch.set_num_threads(2)
set_seed(42)
print('Python:', platform.python_version())
print({p: importlib.metadata.version(p) for p in ['torch','transformers','huggingface-hub','safetensors']})
print('Device: CPU; generation uses greedy decoding; other inference runs in evaluation mode.')

Python: 3.12.14
{'torch': '2.9.1+cpu', 'transformers': '4.57.1', 'huggingface-hub': '0.36.2', 'safetensors': '0.8.0'}
Device: CPU; generation uses greedy decoding; other inference runs in evaluation mode.


### Explicit checkpoints and revisions
Every checkpoint is resolved to an immutable commit. The saved revision file pins reruns.
Public model cards describe the source tasks and limitations. Model-specific code execution is disabled.

In [2]:
MODELS = {
    'sentiment': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
    'mask': 'distilbert/distilbert-base-uncased',
    'ner': 'dslim/bert-base-NER',
    'qa': 'distilbert/distilbert-base-cased-distilled-squad',
    'generation': 'distilbert/distilgpt2'
}
revision_path = Path('assignment31_model_revisions.json')
if revision_path.exists():
    REVISIONS = json.loads(revision_path.read_text())
else:
    REVISIONS = {key: HfApi().model_info(repo).sha for key, repo in MODELS.items()}
    revision_path.write_text(json.dumps(REVISIONS, indent=2))
for key, repo in MODELS.items():
    print(key, repo, REVISIONS[key])
    print('Model card:', 'https://huggingface.co/' + repo)
def load_pipeline(task, key, **kwargs):
    return pipeline(task, model=MODELS[key], revision=REVISIONS[key],
                    device=-1, trust_remote_code=False,
                    model_kwargs={'use_safetensors': True}, **kwargs)
def serialisable(value):
    if isinstance(value, dict): return {k: serialisable(v) for k,v in value.items()}
    if isinstance(value, list): return [serialisable(v) for v in value]
    if hasattr(value, 'item'): return value.item()
    return value
results_record = {}

sentiment distilbert/distilbert-base-uncased-finetuned-sst-2-english 714eb0fa89d2f80546fda750413ed43d93601a13
Model card: https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
mask distilbert/distilbert-base-uncased 12040accade4e8a0f71eabdb258fecc2e7e948be
Model card: https://huggingface.co/distilbert/distilbert-base-uncased
ner dslim/bert-base-NER d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc
Model card: https://huggingface.co/dslim/bert-base-NER
qa distilbert/distilbert-base-cased-distilled-squad 564e9b582944a57a3e586bbb98fd6f0a4118db7f
Model card: https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad
generation distilbert/distilgpt2 2290a62682d06624634c1f46a6ad5be0f47f38aa
Model card: https://huggingface.co/distilbert/distilgpt2


## Part A NLP pipelines
### Sentiment analysis
Classifies each text into the checkpoint's positive or negative labels. Example application:
triaging feedback. The first two inputs are the exact supplied notebook examples.

In [3]:
classifier = load_pipeline('sentiment-analysis', 'sentiment')
texts = ['The lecture was engaging and easy to follow.', 'The notebook is confusing.']
sentiment_results = classifier(texts)
results_record['sentiment'] = list(zip(texts, sentiment_results))
for text, result in zip(texts, sentiment_results): print(text, result)

Device set to use cpu


The lecture was engaging and easy to follow. {'label': 'POSITIVE', 'score': 0.9996544122695923}
The notebook is confusing. {'label': 'NEGATIVE', 'score': 0.999387264251709}


### Sentiment boundary cases
Mixed sentiment and sarcasm challenge a binary label system. These are probes, not a scored test set.

In [4]:
probes = ['The explanations were helpful, but the audio was terrible.',
          'Great, another crash right before my deadline.']
results_record['sentiment_probes'] = list(zip(probes, classifier(probes)))
for item in results_record['sentiment_probes']: print(item)

('The explanations were helpful, but the audio was terrible.', {'label': 'NEGATIVE', 'score': 0.9980430603027344})
('Great, another crash right before my deadline.', {'label': 'POSITIVE', 'score': 0.9729446172714233})


### Fill mask
Predicts plausible replacements for a masked token using bidirectional context.
This supports contextual word completion; a high score does not verify a factual claim.

In [5]:
masker = load_pipeline('fill-mask', 'mask')
mask_input = 'Paris is the [MASK] of France.'
mask_results = masker(mask_input, top_k=3)
results_record['fill_mask'] = serialisable(mask_results)
print(json.dumps(results_record['fill_mask'], indent=2))
del masker; _ = gc.collect()

Device set to use cpu


[
  {
    "score": 0.9815373420715332,
    "token": 3007,
    "token_str": "capital",
    "sequence": "paris is the capital of france."
  },
  {
    "score": 0.0033423895947635174,
    "token": 14508,
    "token_str": "birthplace",
    "sequence": "paris is the birthplace of france."
  },
  {
    "score": 0.0010446939850226045,
    "token": 22037,
    "token_str": "northernmost",
    "sequence": "paris is the northernmost of france."
  }
]


### Named entity recognition
Identifies person, organisation and location spans using a token-classification model.
Subword predictions are aggregated into readable spans. Applications include information extraction.

The checkpoint includes unused BERT pooler weights. The token-classification head operates on token representations, so the loader warning about those pooler weights is expected; it does not indicate a missing NER head.

In [6]:
recogniser = load_pipeline('ner', 'ner', aggregation_strategy='simple')
ner_input = 'Sarah works at Microsoft in London.'
ner_results = recogniser(ner_input)
results_record['ner'] = serialisable(ner_results)
print(json.dumps(results_record['ner'], indent=2))
del recogniser; _ = gc.collect()

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cpu


[
  {
    "entity_group": "PER",
    "score": 0.9990736246109009,
    "word": "Sarah",
    "start": 0,
    "end": 5
  },
  {
    "entity_group": "ORG",
    "score": 0.9986039996147156,
    "word": "Microsoft",
    "start": 15,
    "end": 24
  },
  {
    "entity_group": "LOC",
    "score": 0.9994895458221436,
    "word": "London",
    "start": 28,
    "end": 34
  }
]


### Extractive question answering
Selects an answer span from supplied context. This example tests retrieval of an explicit fact,
not open-domain knowledge. This SQuAD checkpoint is not a reliable no-answer detector.

In [7]:
answerer = load_pipeline('question-answering', 'qa')
context = 'The company opened its first warehouse in Singapore in 2018. Its second warehouse opened in Tokyo in 2021.'
question = 'Where did the company open its first warehouse?'
qa_result = answerer(question=question, context=context)
results_record['question_answering'] = serialisable(qa_result)
print(json.dumps(qa_result, indent=2))
assert context[qa_result['start']:qa_result['end']] == qa_result['answer']
del answerer; _ = gc.collect()

Device set to use cpu


{
  "score": 0.9909743070602417,
  "start": 42,
  "end": 51,
  "answer": "Singapore"
}


### Text generation
Continues a prompt autoregressively. Greedy decoding limits sampling variation, but does not
ensure correct or useful content. DistilGPT2 is a base language model, not an instruction assistant.

In [8]:
generator = load_pipeline('text-generation', 'generation')
generation_result = generator('The purpose of machine learning is', max_new_tokens=40,
                              do_sample=False, temperature=1.0, num_return_sequences=1,
                              pad_token_id=generator.tokenizer.eos_token_id)
results_record['text_generation'] = generation_result
print(generation_result[0]['generated_text'])
del generator; _ = gc.collect()

Device set to use cpu


The purpose of machine learning is to learn how to use the data to learn how to use the data to learn how to use the data to learn how to use the data to learn how to use the data to learn how to use


## Part B Tokenisation and sequence classification
### Token IDs tokens masks and decoding
The supplied sentence is retained. A second short sentence illustrates padding in a batch.

In [9]:
checkpoint = MODELS['sentiment']
tokenizer = AutoTokenizer.from_pretrained(checkpoint, revision=REVISIONS['sentiment'], trust_remote_code=False)
sentences = ['This course is excellent.', 'Awful.']
inputs = tokenizer(sentences, padding=True, truncation=True, max_length=128, return_tensors='pt')
print('input_ids:', inputs['input_ids'].tolist())
print('attention_mask:', inputs['attention_mask'].tolist())
for row in inputs['input_ids']:
    print('tokens:', tokenizer.convert_ids_to_tokens(row.tolist()))
    print('decoded with special tokens:', tokenizer.decode(row))
    print('decoded without special tokens:', tokenizer.decode(row, skip_special_tokens=True))
results_record['tokenisation'] = {k:v.tolist() for k,v in inputs.items()}

input_ids: [[101, 2023, 2607, 2003, 6581, 1012, 102], [101, 9643, 1012, 102, 0, 0, 0]]
attention_mask: [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0, 0]]
tokens: ['[CLS]', 'this', 'course', 'is', 'excellent', '.', '[SEP]']
decoded with special tokens: [CLS] this course is excellent. [SEP]
decoded without special tokens: this course is excellent.
tokens: ['[CLS]', 'awful', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]']
decoded with special tokens: [CLS] awful. [SEP] [PAD] [PAD] [PAD]
decoded without special tokens: awful.


### AutoModel representations
AutoModel produces contextual vectors. It does not return sentiment labels by itself.
The base checkpoint is reused from the masked-language task; its masked-token head is omitted.

In [10]:
base_tokenizer = AutoTokenizer.from_pretrained(MODELS['mask'], revision=REVISIONS['mask'])
base_model = AutoModel.from_pretrained(MODELS['mask'], revision=REVISIONS['mask'],
                                     use_safetensors=True, trust_remote_code=False).eval()
base_inputs = base_tokenizer(sentences, padding=True, return_tensors='pt')
with torch.inference_mode():
    hidden = base_model(**base_inputs).last_hidden_state
print('last_hidden_state shape [batch, sequence, hidden]:', list(hidden.shape))
print('First token first eight coordinates:', hidden[0,0,:8].tolist())
results_record['hidden_shape'] = list(hidden.shape)
assert hidden.shape[:2] == base_inputs['input_ids'].shape
del base_model, hidden; _ = gc.collect()

last_hidden_state shape [batch, sequence, hidden]: [2, 7, 768]
First token first eight coordinates: [-0.127089723944664, -0.04285801202058792, 0.05204109847545624, -0.10051850229501724, 0.16300876438617706, -0.2831284701824188, 0.2863354980945587, 0.5416862964630127]


### Logits Softmax and labels
Reuse the sentiment pipeline's trained sequence classifier. Loading it through
AutoModelForSequenceClassification demonstrates the explicit model interface requested.

In [11]:
del classifier; _ = gc.collect()
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,
         revision=REVISIONS['sentiment'], use_safetensors=True, trust_remote_code=False).eval()
with torch.inference_mode():
    outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)
predicted_ids = probabilities.argmax(dim=-1)
predicted_labels = [model.config.id2label[int(i)] for i in predicted_ids]
print('id2label:', model.config.id2label)
print('logits:', outputs.logits.tolist())
print('probabilities:', probabilities.tolist())
print('predicted labels:', predicted_labels)
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(len(sentences)), atol=1e-6)
assert (probabilities >= 0).all()
results_record['classification'] = {'logits': outputs.logits.tolist(),
                                  'probabilities': probabilities.tolist(), 'labels': predicted_labels}
_ = Path('assignment31_results.json').write_text(json.dumps(serialisable(results_record), indent=2))

id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
logits: [[-4.263468265533447, 4.627264022827148], [4.706772327423096, -3.793555736541748]]
probabilities: [[0.00013763993047177792, 0.9998623132705688], [0.9997966885566711, 0.00020336027955636382]]
predicted labels: ['POSITIVE', 'NEGATIVE']


### Reflection

Tokenisation converts text into vocabulary IDs a model can process, usually using subword units to balance vocabulary size and coverage of unfamiliar words. The notebook displays the IDs, token strings and attention masks, and decodes them back to text. Decoding may normalise case and spacing; it need not reproduce the original bytes.

For the BERT-style tokeniser used here, [CLS] is the sequence-start classification token, and [SEP] marks the end of a sequence or separates a sentence pair. [PAD] fills shorter sequences in a batch. An attention-mask value of 1 marks a real token and 0 marks padding to be ignored. This padding mask differs from a causal decoder mask. DistilBERT retains these token conventions, though its architecture differs from BERT.

AutoModel returns contextual hidden representations, not task labels. AutoModelForSequenceClassification adds a trained classification head. Logits are unrestricted class scores; Softmax converts the mutually exclusive class scores to nonnegative values that sum to one. The predicted label comes from argmax and the checkpoint's id2label mapping. These probabilities express the model's distribution over its labels; a high score does not prove correctness or calibration.

## Part C Transformer architectures

BERT is an encoder-only Transformer. Its self-attention can use tokens on both sides of a position, allowing contextual representations suited to classification, named entity recognition and extractive question answering. Masked-language pretraining teaches it to recover hidden tokens; it is not inherently a left-to-right text generator.

GPT is a decoder-only Transformer. A causal attention mask prevents each position from attending to future tokens during next-token prediction. At inference, generated tokens are appended to the context and the model continues autoregressively. This architecture is therefore a natural choice for text generation. Modern decoder-only models need not contain the encoder cross-attention found in the original encoder-decoder Transformer.

For sentiment analysis, I would start with an encoder and a sequence-classification head. For text generation, I would use a causal decoder. For translation, an encoder-decoder model is a strong default: the encoder represents the source sentence and the decoder generates the target sentence while cross-attending to that representation. Translation does not strictly require this architecture; suitably trained decoder-only models can also translate.

Attention computes compatibility between queries and keys, normalises the scores and uses them to combine value vectors. Scaled dot-product attention divides query–key dot products by the square root of key dimensionality, applies Softmax and forms a weighted sum of values. Multiple heads can capture different relationships. Positional information supplies ordering that attention alone does not encode. Attention weights are useful computational signals, but should not automatically be treated as faithful explanations of a prediction [1, 2].

## Part D Critical evaluation

Pretrained Transformer models provide reusable language representations and reduce the amount of task-specific training needed for an initial application. Hugging Face pipelines package tokenisation, model inference and post-processing into a consistent interface. This makes sentiment classification, masked-token prediction, entity extraction, question answering and text continuation accessible without building an architecture from scratch. The accompanying notebook demonstrates these tasks using explicit checkpoints, while preserving the supplied sentiment examples and sequence-classification sentence. These examples establish functionality, rather than general business accuracy.

A major limitation is the gap between pretraining or fine-tuning data and the deployment domain. A sentiment model trained on movie reviews may handle ordinary praise or criticism but misread technical complaints, mixed sentiment or sarcasm. A model with only positive and negative labels cannot faithfully represent every neutral or uncertain statement. Named entity labels are also constrained by training conventions. Pipeline confidence scores should therefore be evaluated on representative labelled data, not interpreted as universal measures of truth.

Hallucination is particularly relevant to generation. A causal language model predicts plausible continuations; it does not independently verify facts. Fluent text may invent events, explanations or sources. Extractive question answering narrows the output to a span in the supplied context, but can still choose an incorrect span or answer a question the context does not support. A model trained without a no-answer option needs an additional abstention strategy. Human review, source grounding and task-specific evaluation remain necessary.

Bias can arise from imbalanced language, demographic representation, annotation choices and historical stereotypes. The same sentence structure may produce different predictions when names or group references change. Testing should include relevant dialects, languages and counterfactual examples, while avoiding claims that a small fairness checklist eliminates bias. Privacy and consent also matter: sensitive customer text should not be placed in public demonstrations or retained indefinitely in logs. Local inference reduces transmission but does not remove access-control responsibilities.

Apparent understanding should be assessed through behaviour. Good performance on familiar phrasing can coexist with failures on negation, unfamiliar contexts or compositional questions. This does not mean that models only copy text, but it cautions against equating fluent output with dependable reasoning. Truncation and limited context can remove decisive information. Even deterministic decoding does not ensure correctness. In this run, sarcasm was labelled positive with a score of 0.9729, and greedy DistilGPT2 repeatedly continued with “learn how to use the data”. These concrete failures show why high confidence and deterministic output are insufficient.

Resource requirements include model downloads, memory, tokenisation time and inference compute. Larger models may increase latency and serving costs. Task-specific encoders can be preferable to general generators for bounded classification; batching and smaller checkpoints may improve efficiency, subject to quality checks.

Fine-tuning is appropriate when a stable task, domain vocabulary or label scheme is poorly served by the baseline and sufficient high-quality examples are available. It should use separated training, validation and test sets, documented provenance and subgroup evaluation. Retrieval is often preferable for frequently changing factual knowledge. Fine-tuning is not a universal cure for hallucination or bias. Deployment should retain human escalation, transparent limitations, versioned models and monitoring, with rollback when performance deteriorates.

## Checks and limits
- All five pipelines use explicit public checkpoints and saved revisions.
- The QA span is checked against its source context; probability rows are checked to sum to one.
- Input IDs and attention masks are displayed; padding and special tokens are visible.
- Examples demonstrate operation only. No held-out business accuracy, bias score or latency claim is made.
- Review printed probe and generation outputs before deciding whether these models fit an application.

## References

[1] Vaswani, A. et al. (2017). Attention Is All You Need. https://arxiv.org/abs/1706.03762

[2] Devlin, J. et al. (2018). BERT Pre-training of Deep Bidirectional Transformers for Language Understanding. https://arxiv.org/abs/1810.04805

[3] Hugging Face. Transformers 4.57.1 pipeline documentation. https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines

[4] Hugging Face. DistilBERT SST-2 model card. https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english

Additional model cards and immutable revisions are recorded in the notebook.